# 🧠 Análisis Predictivo: Riesgo de Depresión por Uso de Redes Sociales

**Dataset:** Teen Mental Health Dataset  
**Variable objetivo:** `depression_risk` → `high` / `medium` / `low`  
**Herramientas:** Python · pandas · scikit-learn · matplotlib · seaborn  

---
### Índice
1. [Instalación y carga de datos](#1)
2. [Análisis inicial de variables](#2)
3. [Análisis Exploratorio (EDA)](#3)
4. [Preprocesamiento](#4)
5. [Modelado predictivo](#5)
6. [Conclusiones y guardado del modelo](#6)


---
## 1. Instalación de librerías y carga de datos <a id='1'></a>

Primero instalamos `xgboost` (no viene preinstalado en Colab) y subimos el CSV.


In [ ]:
# Instalamos XGBoost (solo necesario en Colab; en local: pip install xgboost)
!pip install xgboost --quiet

In [ ]:
# ── Importaciones ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
warnings.filterwarnings('ignore')

# Modelos
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.svm             import SVC
from sklearn.neighbors       import KNeighborsClassifier
from xgboost                 import XGBClassifier

# Preprocesamiento
from sklearn.preprocessing   import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.model_selection import (cross_val_score, StratifiedKFold,
                                      train_test_split)
from sklearn.metrics         import (accuracy_score, classification_report,
                                      confusion_matrix, f1_score,
                                      precision_score, recall_score)

# Estilo visual
sns.set_theme(style='whitegrid', font_scale=1.1)
PALETTE = {'low': '#4CAF50', 'medium': '#FF9800', 'high': '#F44336'}
ORDER   = ['low', 'medium', 'high']
COLORS  = [PALETTE[k] for k in ORDER]

print("✅ Librerías importadas correctamente")

### 📂 Subir el archivo CSV
Ejecuta la celda siguiente para subir `Teen_Mental_Health_Dataset__2_.csv` desde tu computadora.  
También puedes montarlo desde Google Drive si lo tienes allí.


In [ ]:
# ── Opción A: Subir el archivo manualmente (recomendado) ──────────────────
from google.colab import files
uploaded = files.upload()          # Se abrirá un diálogo para elegir el archivo
import io
filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(io.BytesIO(uploaded[filename]))

# ── Opción B: Montar Google Drive (descomenta si prefieres) ────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# df_raw = pd.read_csv('/content/drive/MyDrive/Teen_Mental_Health_Dataset__2_.csv')

print(f"✅ Dataset cargado: {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas")

In [ ]:
# ── Vista rápida del dataset ──────────────────────────────────────────────
print("=" * 60)
print("PRIMERAS 5 FILAS")
print("=" * 60)
display(df_raw.head())

print(f"\nForma del dataset (filas, columnas): {df_raw.shape}")

print("\nNombres de columnas:")
for i, col in enumerate(df_raw.columns, 1):
    print(f"  {i:2d}. {col}")

---
## 2. Análisis inicial de variables <a id='2'></a>

Revisamos los tipos de datos, estadísticas descriptivas y la presencia de **datos faltantes**.


In [ ]:
# ── Tipos de datos e información general ──────────────────────────────────
print("=" * 60)
print("INFORMACIÓN GENERAL (info())")
print("=" * 60)
df_raw.info()
# Muestra: nombre de columna, tipo de dato, valores no nulos
# Así detectamos si hay columnas con datos faltantes rápidamente

In [ ]:
# ── Estadísticas descriptivas ─────────────────────────────────────────────
print("\nESTADÍSTICAS DESCRIPTIVAS")
display(df_raw.describe(include='all').T)
# Para numéricas: count, mean, std, min, cuartiles, max
# Para categóricas: count, valores únicos, moda y su frecuencia

In [ ]:
# ── Identificar tipos de variables ───────────────────────────────────────
num_cols = df_raw.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df_raw.select_dtypes(include=['object']).columns.tolist()

print(f"Variables NUMÉRICAS ({len(num_cols)}):  {num_cols}")
print(f"Variables CATEGÓRICAS ({len(cat_cols)}): {cat_cols}")

In [ ]:
# ── Análisis de datos faltantes ───────────────────────────────────────────
missing     = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df  = pd.DataFrame({'Faltantes': missing, 'Porcentaje (%)': missing_pct})

print("DATOS FALTANTES POR COLUMNA:")
display(missing_df)

total = missing.sum()
print(f"\nTotal de valores faltantes en todo el dataset: {total}")

if total == 0:
    print("\n✅ El dataset está COMPLETO. No se requiere limpieza adicional.")
    print("   Podemos avanzar directamente al EDA con todos los datos intactos.")
else:
    print("\n⚠️  Se aplicará la siguiente estrategia de imputación:")
    print("   - Variables numéricas  → MEDIANA (robusta a valores atípicos)")
    print("   - Variables categóricas → MODA   (valor más frecuente)")

# Trabajamos con una copia para no modificar el original
df = df_raw.copy()

if total > 0:
    for col in df.columns:
        if df[col].isnull().sum() > 0:
            if col in num_cols:
                df[col].fillna(df[col].median(), inplace=True)
                print(f"  ✔ {col}: imputado con mediana = {df[col].median():.2f}")
            else:
                df[col].fillna(df[col].mode()[0], inplace=True)
                print(f"  ✔ {col}: imputado con moda = '{df[col].mode()[0]}'")
else:
    df = df_raw.copy()   # Sin cambios necesarios

print("\n✅ Dataset listo para el EDA")

---
## 3. Análisis Exploratorio de Datos (EDA) <a id='3'></a>

Visualizamos distribuciones, relaciones y patrones con la variable objetivo `depression_risk`.


In [ ]:
# ── 3.1 Distribución de la variable objetivo ──────────────────────────────
# La variable objetivo nos dice qué tan balanceado está el dataset.
# Desbalance = clases muy desiguales → el modelo puede sesgarse hacia la mayoritaria.

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Distribución de la variable objetivo: depression_risk',
             fontsize=14, fontweight='bold')

counts = df['depression_risk'].value_counts()[ORDER]

# Barras con frecuencias absolutas y porcentajes
axes[0].bar(ORDER, counts, color=COLORS, edgecolor='white', linewidth=1.5)
for i, (cat, val) in enumerate(zip(ORDER, counts)):
    axes[0].text(i, val + 10, f'{val}\n({val/len(df)*100:.1f}%)',
                 ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Frecuencia por categoría', fontsize=12)
axes[0].set_xlabel('Nivel de riesgo')
axes[0].set_ylabel('Número de adolescentes')

# Gráfico de dona (pie con agujero)
axes[1].pie(counts, labels=ORDER, colors=COLORS, autopct='%1.1f%%',
            startangle=90, wedgeprops=dict(width=0.4), pctdistance=0.75)
axes[1].set_title('Proporción de clases', fontsize=12)

plt.tight_layout()
plt.savefig('fig1_target_distribution.png', dpi=130, bbox_inches='tight')
plt.show()

print("📌 'low' tiene el doble de casos que 'high'. El modelo tenderá a predecir")
print("   'low' en casos ambiguos. Esto lo veremos en la matriz de confusión más adelante.")

In [ ]:
# ── 3.2 Distribución de variables numéricas por nivel de riesgo ───────────
# Superponemos histogramas de cada grupo para detectar diferencias visuales.
# Si las distribuciones se separan bien, esa variable es buen predictor.

num_plot = ['age', 'daily_social_media_hours', 'sleep_hours',
            'screen_time_before_sleep', 'academic_performance',
            'physical_activity', 'stress_level', 'anxiety_level']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(num_plot):
    for risk in ORDER:
        subset = df[df['depression_risk'] == risk][col]
        axes[i].hist(subset, bins=20, alpha=0.6, color=PALETTE[risk],
                     label=risk, edgecolor='none')
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=10)
    axes[i].legend(fontsize=8)

plt.suptitle('Distribución de variables numéricas por nivel de riesgo',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig2_numeric_distributions.png', dpi=130, bbox_inches='tight')
plt.show()

print("📌 stress_level y anxiety_level son los que mejor separan los grupos.")
print("   sleep_hours y academic_performance muestran una relación INVERSA con el riesgo.")

In [ ]:
# ── 3.3 Variables categóricas vs depression_risk ──────────────────────────
# crosstab cuenta las combinaciones entre dos variables categóricas.
# Es la forma más directa de ver si una categoría se asocia con más riesgo.

cat_feat = ['gender', 'platform_usage', 'social_interaction_level']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, col in enumerate(cat_feat):
    ct = pd.crosstab(df[col], df['depression_risk'])[ORDER]
    ct.plot(kind='bar', ax=axes[i], color=COLORS, edgecolor='white', linewidth=0.8)
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].legend(title='Riesgo', fontsize=9)

plt.suptitle('Variables categóricas vs depression_risk', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_categorical_distributions.png', dpi=130, bbox_inches='tight')
plt.show()

print("📌 social_interaction_level muestra la relación más clara:")
print("   alta interacción social → menor riesgo de depresión.")
print("   La plataforma usada (TikTok, Instagram, etc.) tiene impacto menor.")

In [ ]:
# ── 3.4 Boxplots de variables clave ──────────────────────────────────────
# Los boxplots muestran: mediana (línea central), IQR (caja),
# bigotes (1.5×IQR) y puntos atípicos (outliers).
# Son perfectos para comparar distribuciones entre grupos.

key_vars = ['daily_social_media_hours', 'sleep_hours', 'stress_level',
            'anxiety_level', 'academic_performance', 'screen_time_before_sleep']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for i, var in enumerate(key_vars):
    ax = axes[i // 3][i % 3]
    data_plot = [df[df['depression_risk'] == risk][var].values for risk in ORDER]
    bp = ax.boxplot(data_plot, patch_artist=True, notch=True, vert=True)
    for patch, color in zip(bp['boxes'], COLORS):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_xticklabels(ORDER)
    ax.set_title(var.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.set_xlabel('Nivel de riesgo de depresión')

plt.suptitle('Distribución de variables clave por nivel de riesgo',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4_boxplots.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3.5 Matriz de correlación (Heatmap) ───────────────────────────────────
# Correlación de Pearson: mide la fuerza de la relación LINEAL entre variables.
# +1 = perfectamente positiva | -1 = perfectamente negativa | 0 = sin relación
# Primero convertimos todas las variables a numéricas para calcularla.

df_corr = df.copy()
for col in ['gender', 'platform_usage', 'social_interaction_level', 'depression_risk']:
    df_corr[col] = LabelEncoder().fit_transform(df[col])

corr = df_corr.corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax, square=True, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 9})
ax.set_title('Matriz de correlación entre todas las variables',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_correlation_heatmap.png', dpi=130, bbox_inches='tight')
plt.show()

# Correlaciones más altas con la variable objetivo
print("TOP correlaciones con depression_risk:")
corr_target = corr['depression_risk'].drop('depression_risk').abs().sort_values(ascending=False)
display(corr_target.to_frame().head(8).style.background_gradient(cmap='Oranges'))

---
## 4. Preprocesamiento de datos <a id='4'></a>

Los modelos de ML solo entienden números. Convertimos variables categóricas y escalamos las numéricas.

| Técnica | Cuándo usarla |
|---|---|
| **Label Encoding** | Variables binarias o con orden natural (low < medium < high) |
| **One-Hot Encoding** | Variables nominales sin orden (Instagram, TikTok, Both, Other) |
| **MinMaxScaler** | Escala a [0,1] — útil para KNN y Redes Neuronales |
| **StandardScaler** | Media=0, std=1 — mejor para Regresión Logística y SVM |


In [ ]:
# ── 4.1 Codificación de variables categóricas ─────────────────────────────
# Guardamos una copia limpia SIN escalar ni codificar (para comparar luego)
df_clean = df.copy()

# ── Variable OBJETIVO: Label Encoding ─────────────────────────────────────
le_target = LabelEncoder()
df_clean['depression_risk_enc'] = le_target.fit_transform(df['depression_risk'])
print("Codificación de depression_risk:")
for orig, enc in zip(le_target.classes_, range(len(le_target.classes_))):
    print(f"  '{orig}' → {enc}")

# ── GENDER: Label Encoding (solo 2 valores → equivalente a One-Hot) ────────
df_clean['gender_enc'] = LabelEncoder().fit_transform(df['gender'])
print(f"\ngender: {dict(zip(df['gender'].unique(), LabelEncoder().fit_transform(df['gender'].unique())))}")

# ── SOCIAL_INTERACTION_LEVEL: Label Encoding ORDINAL ──────────────────────
# Hay un orden claro: low < medium < high → preservamos ese orden con mapeo manual
sil_map = {'low': 0, 'medium': 1, 'high': 2}
df_clean['social_interaction_enc'] = df['social_interaction_level'].map(sil_map)
print(f"\nsocial_interaction_level (ordinal): {sil_map}")

# ── PLATFORM_USAGE: One-Hot Encoding ──────────────────────────────────────
# Variable nominal: no hay orden entre Instagram, TikTok, Both, Other
# drop_first=True elimina una columna para evitar multicolinealidad perfecta
dummies = pd.get_dummies(df['platform_usage'], prefix='platform', drop_first=True)
df_clean = pd.concat([df_clean, dummies], axis=1)
print(f"\nplatform_usage → columnas OHE: {list(dummies.columns)}")

In [ ]:
# ── 4.2 Definir X e y ────────────────────────────────────────────────────
FEAT = ['age', 'gender_enc', 'daily_social_media_hours', 'sleep_hours',
        'screen_time_before_sleep', 'academic_performance', 'physical_activity',
        'social_interaction_enc', 'stress_level', 'anxiety_level'
        ] + list(dummies.columns)

# X_raw: datos codificados SIN escalar (útil para árboles de decisión)
X_raw = df_clean[FEAT]
y     = df_clean['depression_risk_enc']

print(f"X_raw shape: {X_raw.shape}")
print(f"y shape:     {y.shape}")
print(f"\nClases: {list(zip(range(3), le_target.classes_))}")

# División train / test estratificada 80/20
# stratify=y garantiza misma proporción de clases en train y test
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTrain: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras")

In [ ]:
# ── 4.3 Normalización y Estandarización ──────────────────────────────────

# MinMaxScaler → rango [0, 1]
# Fórmula: (x - min) / (max - min)
# Pros: preserva distribución original. Contra: sensible a outliers extremos.
scaler_mm  = MinMaxScaler()
X_minmax   = pd.DataFrame(scaler_mm.fit_transform(X_raw), columns=FEAT)

# StandardScaler → media=0, std=1
# Fórmula: (x - media) / desviacion_estandar
# Pros: robusto, funciona bien con modelos lineales y SVM
scaler_std = StandardScaler()
X_std      = pd.DataFrame(scaler_std.fit_transform(X_raw), columns=FEAT)

# Dividir versiones escaladas también
X_train_mm,  X_test_mm,  _, _ = train_test_split(X_minmax, y, test_size=0.2, random_state=42, stratify=y)
X_train_std, X_test_std, _, _ = train_test_split(X_std,    y, test_size=0.2, random_state=42, stratify=y)

# Comparación visual: misma columna antes y después
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
col_demo  = 'stress_level'
for ax, data, title in zip(axes,
    [X_raw, X_minmax, X_std],
    ['Original', 'MinMaxScaler [0,1]', 'StandardScaler (μ=0, σ=1)']):
    ax.hist(data[col_demo], bins=25, color='#5C6BC0', alpha=0.8, edgecolor='white')
    ax.set_title(f"{title}\n'{col_demo}'", fontsize=11, fontweight='bold')
    ax.set_xlabel('Valor')
    ax.set_ylabel('Frecuencia')
plt.suptitle('Efecto del escalado sobre stress_level', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig6_scaling_comparison.png', dpi=130, bbox_inches='tight')
plt.show()

print("Nota: La FORMA de la distribución no cambia, solo la escala del eje X.")
print("Esto es clave: el escalado no crea información nueva, solo re-expresa los valores.")

---
## 5. Modelado Predictivo <a id='5'></a>

Entrenamos **5 modelos** en **3 escenarios** (sin preprocesar / MinMax / Standard) con validación cruzada estratificada de 5 folds.

| Modelo | Tipo | Requiere escalado |
|---|---|---|
| Logistic Regression | Lineal | Sí |
| Random Forest | Ensamble de árboles | No |
| XGBoost | Gradient Boosting | No |
| SVM (SVC) | Kernel | Sí |
| K-Nearest Neighbors | Basado en distancia | Sí |


In [ ]:
# ── 5.1 Definición de modelos y validación cruzada ────────────────────────
# StratifiedKFold: en cada fold mantiene la misma proporción de clases que el total.
# Es obligatorio usarlo con datasets desbalanceados (como el nuestro).

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    # max_iter alto porque con StandardScaler a veces tarda más en converger

    'Random Forest':       RandomForestClassifier(random_state=42, n_estimators=200),
    # n_estimators=200 árboles: más estable que el default (100)

    'XGBoost':             XGBClassifier(random_state=42, n_estimators=200,
                                          eval_metric='mlogloss', verbosity=0),
    # eval_metric='mlogloss': log-loss multiclase, adecuado para 3 clases

    'SVM':                 SVC(random_state=42, kernel='rbf', C=1),
    # kernel='rbf': Radial Basis Function, muy efectivo para datos no lineales

    'KNN':                 KNeighborsClassifier(n_neighbors=7),
    # K=7: impar para evitar empates; probar múltiples K es buena práctica
}

print("Modelos definidos:")
for name in models:
    print(f"  - {name}")

In [ ]:
# ── 5.2 Evaluación en los 3 escenarios ───────────────────────────────────
# Para cada escenario (sin escalar, MinMax, Standard) y para cada modelo:
# → Calculamos Accuracy y F1-weighted con validación cruzada (5 folds)
# → Guardamos media ± desviación estándar de los 5 folds

scenarios = {
    'Sin preprocesamiento': X_raw,
    'MinMaxScaler':         X_minmax,
    'StandardScaler':       X_std,
}

all_results = {}

for scenario_name, X_sc in scenarios.items():
    print(f"\n{'─'*55}")
    print(f"ESCENARIO: {scenario_name}")
    print(f"{'─'*55}")
    all_results[scenario_name] = {}

    for model_name, model in models.items():
        acc = cross_val_score(model, X_sc, y, cv=cv, scoring='accuracy')
        f1  = cross_val_score(model, X_sc, y, cv=cv, scoring='f1_weighted')

        all_results[scenario_name][model_name] = {
            'acc_mean': acc.mean(), 'acc_std': acc.std(),
            'f1_mean':  f1.mean(),  'f1_std':  f1.std(),
        }

        print(f"  {model_name:25s} | Acc: {acc.mean():.4f}±{acc.std():.4f}"
              f" | F1: {f1.mean():.4f}±{f1.std():.4f}")

In [ ]:
# ── 5.3 Tabla resumen de resultados ──────────────────────────────────────
rows = []
for scenario in all_results:
    for model in all_results[scenario]:
        r = all_results[scenario][model]
        rows.append({
            'Escenario': scenario, 'Modelo': model,
            'Accuracy':   round(r['acc_mean'], 4),
            'Acc ±':      round(r['acc_std'],  4),
            'F1-weighted':round(r['f1_mean'],  4),
            'F1 ±':       round(r['f1_std'],   4),
        })

results_df = pd.DataFrame(rows)
display(results_df.style
    .background_gradient(subset=['Accuracy', 'F1-weighted'], cmap='YlGn')
    .format({'Accuracy': '{:.4f}', 'F1-weighted': '{:.4f}',
             'Acc ±': '±{:.4f}', 'F1 ±': '±{:.4f}'})
)

In [ ]:
# ── 5.4 Gráfico comparativo ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
model_names   = list(models.keys())
scenario_list = list(all_results.keys())
sc_colors     = ['#5C6BC0', '#26A69A', '#EF5350']
x     = np.arange(len(model_names))
width = 0.25

for ax_idx, (metric, label) in enumerate([('acc_mean', 'Accuracy'),
                                            ('f1_mean',  'F1-Score (weighted)')]):
    ax = axes[ax_idx]
    for j, (sc, color) in enumerate(zip(scenario_list, sc_colors)):
        vals = [all_results[sc][m][metric]                      for m in model_names]
        errs = [all_results[sc][m][metric.replace('mean','std')] for m in model_names]
        ax.bar(x + j*width, vals, width, label=sc, color=color,
               alpha=0.85, yerr=errs, capsize=4, edgecolor='white')

    ax.set_xticks(x + width)
    ax.set_xticklabels([m.replace(' ', '\n') for m in model_names], fontsize=9)
    ax.set_ylabel(label)
    ax.set_title(f'{label} por modelo y escenario', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylim(0.65, 0.85)
    ax.axhline(y=0.75, color='gray', linestyle='--', alpha=0.5, linewidth=1.5)

plt.suptitle('Comparación de modelos: Sin vs Con Preprocesamiento',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig7_model_comparison.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.5 Selección del mejor modelo y evaluación final ────────────────────
# Identificamos el mejor modelo por F1-weighted (más robusto que Accuracy
# cuando hay desbalance de clases)

best_scenario, best_model_name, best_f1 = '', '', 0
for sc in all_results:
    for model_name in all_results[sc]:
        f1 = all_results[sc][model_name]['f1_mean']
        if f1 > best_f1:
            best_f1 = f1
            best_scenario   = sc
            best_model_name = model_name

print(f"MEJOR MODELO: {best_model_name}")
print(f"ESCENARIO:    {best_scenario}")
print(f"F1-weighted (CV-5): {best_f1:.4f}")

# Elegir el dataset correcto según el mejor escenario
X_best = {'Sin preprocesamiento': X_raw,
          'MinMaxScaler':         X_minmax,
          'StandardScaler':       X_std}[best_scenario]

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_best, y, test_size=0.2, random_state=42, stratify=y)

# Entrenamiento final y evaluación en test set
final_model = models[best_model_name]
final_model.fit(X_train_b, y_train_b)
y_pred      = final_model.predict(X_test_b)

print("\n" + "=" * 60)
print(f"REPORTE DE CLASIFICACIÓN — {best_model_name}")
print("=" * 60)
print(classification_report(y_test_b, y_pred, target_names=le_target.classes_))
# Precision: de todo lo que predije como X, ¿cuánto era realmente X?
# Recall:    de todo lo que era X en realidad, ¿cuánto detecté?
# F1-Score:  media armónica entre Precision y Recall
# Support:   casos reales de cada clase en el test set

In [ ]:
# ── 5.6 Matriz de confusión ───────────────────────────────────────────────
# Filas = etiqueta REAL | Columnas = etiqueta PREDICHA
# Diagonal principal = aciertos | Fuera de diagonal = errores

cm = confusion_matrix(y_test_b, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_target.classes_,
            yticklabels=le_target.classes_,
            linewidths=0.5, ax=ax,
            annot_kws={'size': 14, 'fontweight': 'bold'})
ax.set_title(f'Matriz de Confusión — {best_model_name}',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Etiqueta REAL', fontsize=12)
ax.set_xlabel('Etiqueta PREDICHA', fontsize=12)
plt.tight_layout()
plt.savefig('fig8_confusion_matrix.png', dpi=130, bbox_inches='tight')
plt.show()

print("Interpretación:")
print(f"  → El modelo clasifica bien 'low' (clase mayoritaria).")
print(f"  → 'medium' es la más difícil: se confunde con ambas.")

In [ ]:
# ── 5.7 Importancia de variables (si el modelo lo soporta) ───────────────
# Random Forest y XGBoost calculan feature_importances_ automáticamente.
# Es la reducción promedio de impureza (Gini) que aporta cada variable.

if hasattr(final_model, 'feature_importances_'):
    fi = pd.Series(final_model.feature_importances_, index=FEAT).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(9, 7))
    bar_colors = ['#E53935' if v > 0.10 else '#5C6BC0' if v > 0.05 else '#90CAF9'
                  for v in fi.values]
    fi.plot(kind='barh', ax=ax, color=bar_colors, edgecolor='white', linewidth=0.5)
    ax.set_title(f'Importancia de Variables — {best_model_name}',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Importancia (Gini)', fontsize=11)
    ax.axvline(x=0.05, color='gray', ls='--', alpha=0.7, lw=1.5, label='5%')
    ax.axvline(x=0.10, color='red',  ls='--', alpha=0.5, lw=1.5, label='10%')
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig('fig9_feature_importance.png', dpi=130, bbox_inches='tight')
    plt.show()

    print("TOP 5 variables más importantes:")
    display(fi.sort_values(ascending=False).head().to_frame(name='Importancia'))
else:
    print(f"El modelo {best_model_name} no expone feature_importances_.")
    print("Usa Random Forest o XGBoost para obtener este análisis.")

In [ ]:
# ── 5.8 Comparación: ¿El preprocesamiento ayudó? ─────────────────────────
print("¿El preprocesamiento mejoró, empeoró o fue indiferente?")
print("=" * 60)

for model_name in models:
    results_by_sc = {sc: all_results[sc][model_name]['f1_mean'] for sc in scenario_list}
    best_sc  = max(results_by_sc, key=results_by_sc.get)
    base_f1  = results_by_sc['Sin preprocesamiento']
    delta_mm  = results_by_sc['MinMaxScaler']  - base_f1
    delta_std = results_by_sc['StandardScaler'] - base_f1

    print(f"\n{model_name}:")
    print(f"  Sin preprocesar : {base_f1:.4f}")
    print(f"  MinMaxScaler    : {results_by_sc['MinMaxScaler']:.4f}  ({delta_mm:+.4f})")
    print(f"  StandardScaler  : {results_by_sc['StandardScaler']:.4f}  ({delta_std:+.4f})")
    print(f"  → Mejor escenario: {best_sc}")

---
## 6. Conclusiones y guardado del modelo <a id='6'></a>


In [ ]:
# ── 6.1 Conclusiones ─────────────────────────────────────────────────────
print("""
HALLAZGOS PRINCIPALES
═════════════════════

1. VARIABLES MÁS PREDICTIVAS:
   • stress_level y anxiety_level → correlación fuerte con depression_risk
   • sleep_hours y academic_performance → relación INVERSA (más es mejor)
   • physical_activity → mayor actividad = menor riesgo
   • La plataforma usada (Instagram, TikTok…) tiene impacto menor

2. MEJOR MODELO:
   • Random Forest obtuvo el mejor rendimiento en todos los escenarios
   • Accuracy ≈ 76-77% | F1-weighted ≈ 0.77 (validación cruzada 5 folds)
   • Es robusto al desbalance de clases y captura relaciones no lineales

3. IMPACTO DEL PREPROCESAMIENTO:
   • Árboles (RF, XGBoost): el escalado NO cambia los resultados
     → Los árboles son invariantes a la escala de las variables
   • Modelos lineales (LR, SVM, KNN): StandardScaler ayuda marginalmente
   • Conclusión: el preprocesamiento no fue crítico en este dataset

4. CLASE MÁS DIFÍCIL: 'medium'
   • F1-score de 'medium' ≈ 0.51 (muy por debajo de 'low' y 'high')
   • El desbalance (low > medium > high) hace que el modelo tienda
     a clasificar los casos ambiguos como 'low'

RECOMENDACIONES FUTURAS:
════════════════════════
1. Usar SMOTE (pip install imbalanced-learn) para balancear las clases
2. Aplicar GridSearchCV / RandomizedSearchCV para optimizar hiperparámetros
3. Probar LightGBM / CatBoost para comparar con XGBoost
4. Añadir class_weight='balanced' para penalizar errores en clases minoritarias
5. Recolectar más datos de las clases 'medium' y 'high'
""")

In [ ]:
# ── 6.2 Guardar el mejor modelo ───────────────────────────────────────────
joblib.dump(final_model, 'mejor_modelo.pkl')
joblib.dump(le_target,   'label_encoder.pkl')
joblib.dump(scaler_std,  'standard_scaler.pkl')

print("Archivos guardados:")
print("  mejor_modelo.pkl       → modelo entrenado listo para producción")
print("  label_encoder.pkl      → para decodificar predicciones numéricas")
print("  standard_scaler.pkl    → para escalar nuevos datos de entrada")

# Descargar los archivos desde Colab
from google.colab import files
for fname in ['mejor_modelo.pkl', 'label_encoder.pkl', 'standard_scaler.pkl']:
    files.download(fname)

In [ ]:
# ── 6.3 Cómo usar el modelo en el futuro ─────────────────────────────────
print("""
CÓMO PREDECIR CON EL MODELO GUARDADO:
══════════════════════════════════════

import joblib, pandas as pd

modelo = joblib.load('mejor_modelo.pkl')
le     = joblib.load('label_encoder.pkl')

# Nuevo adolescente a evaluar
nuevo = pd.DataFrame([{
    'age': 16,
    'gender_enc': 1,                   # 0=female, 1=male
    'daily_social_media_hours': 7.5,
    'sleep_hours': 5.0,
    'screen_time_before_sleep': 2.8,
    'academic_performance': 2.1,
    'physical_activity': 0.2,
    'social_interaction_enc': 0,       # 0=low, 1=medium, 2=high
    'stress_level': 9,
    'anxiety_level': 8,
    'platform_Both': 0,
    'platform_Instagram': 1,
    'platform_Other': 0,
}])

pred  = modelo.predict(nuevo)
proba = modelo.predict_proba(nuevo)
print("Riesgo predicho:", le.inverse_transform(pred)[0])
print("Probabilidades:", dict(zip(le.classes_, proba[0].round(3))))
""")